<a href="https://colab.research.google.com/github/nothing248/notebooks/blob/main/notebooks/video_subtitle/remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 查看cuda版本
!nvcc -V
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Tue Jul 21 08:55:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0       

In [37]:
import os
# @title 定义路径
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: 
    # 创建tmp
    os.makedirs('/kaggle/temp', exist_ok=True)
    # Kaggle 环境
    pro_path = '/kaggle/temp/video-subtitle-remover'
    exe_path = pro_path
    input_path = '/kaggle/input/datasets/xiaoyuetmyang/subtitle'
    output_path = '/kaggle/working'
elif "COLAB_RELEASE_TAG" in os.environ: 
    # colab 环境
    from google.colab import drive
    drive.mount('/content/drive')
    
    pro_path = '/content/drive/MyDrive/projects/video-subtitle-remover'
    exe_path = '/content/temp'
    input_path = f'{pro_path}/test'
    output_path = input_path

In [17]:
# @title 克隆仓库
!git clone --depth 1 https://github.com/YaoFANGUK/video-subtitle-remover {pro_path}

Cloning into '/kaggle/temp/video-subtitle-remover'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 230 (delta 14), reused 162 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 754.55 MiB | 34.67 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Updating files: 100% (184/184), done.


In [18]:
# @title 安装uv
%cd {exe_path}

import os

!pip install uv
os.environ["UV_VENV_CLEAR"] = "1"
!uv venv

/kaggle/temp/video-subtitle-remover
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 64.3 MB/s eta 0:00:00:00:0100:01
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [20]:
# @title 安装环境
!uv pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ --python .venv/bin/python
!uv pip install torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu118 --python .venv/bin/python
!uv pip install onnxruntime-gpu==1.22.0 --python .venv/bin/python
!uv pip install nvidia-cudnn-cu11==8.6.0.163 --python .venv/bin/python
!uv pip install -r {pro_path}/requirements.txt --python .venv/bin/python
!uv pip install pip setuptools --python .venv/bin/python
!uv run paddlex --install hpi-gpu
!uv pip install matplotlib-inline --python .venv/bin/python
!uv pip install wrapt --python .venv/bin/python

Resolved 26 packages in 4.50s                                        
Uninstalled 2 packages in 1ms
Installed 2 packages in 3ms.9.6.50                          
 - nvidia-cudnn-cu11==8.6.0.163
 + nvidia-cudnn-cu11==8.9.6.50
 - nvidia-nccl-cu11==2.21.5
 + nvidia-nccl-cu11==2.19.3
Resolved 25 packages in 344ms                                        
Uninstalled 2 packages in 2ms
Installed 2 packages in 3ms.1.0.70                          
 - nvidia-cudnn-cu11==8.9.6.50
 + nvidia-cudnn-cu11==9.1.0.70
 - nvidia-nccl-cu11==2.19.3
 + nvidia-nccl-cu11==2.21.5
Checked 1 package in 2ms
Resolved 2 packages in 3ms                                           
Uninstalled 1 package in 1ms
Installed 1 package in 2ms8.6.0.163                         
 - nvidia-cudnn-cu11==9.1.0.70
 + nvidia-cudnn-cu11==8.6.0.163
Resolved 76 packages in 1.50s                                        
Prepared 63 packages in 11.69s                                           
Uninstalled 1 package in 20ms
Installed 63 packag

In [27]:
# @title 查看帮助
%cd {pro_path}
!{exe_path}/.venv/bin/python ./backend/main.py --help

/kaggle/temp/video-subtitle-remover
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'

📢 Tips: QFluentWidgets Pro is now released. Click https://qfluentwidgets.com/pages/pro to learn more about it.

usage: main.py [-h] --input INPUT [--output OUTPUT]
               [--subtitle-area-coords YMIN YMAX XMIN XMAX]
               [--inpaint-mode {sttn-auto,sttn-det,lama,propainter,opencv}]

Video Subtitle Remover Command Line Tool

options:
  -h, --help            show this help message and exit
  --input INPUT, -i INPUT
                        Input video file path
  --output OUTPUT, -o OUTPUT
                        Output video file path (optional)
  --subtitle-area-coords YMIN YMAX XMIN XMAX, -c YMIN YMAX XMIN XMAX
                        Subtitle area coordinates (ymin ymax xmin xmax). Can
                        be specified multiple times for multiple areas.
  --inpaint-mode {sttn-auto,sttn-det,lama,propainter,opencv}
               

In [31]:
!{exe_path}/.venv/bin/python --version

Python 3.12.13


In [45]:
# @title 补丁: GPU字幕检测与无音频兼容
import os

%cd {pro_path}

patch_code = f"""
import sys
import runpy
import subprocess
from paddleocr import TextDetection

# 1. 强制 GPU 模式
_init = TextDetection.__init__
TextDetection.__init__ = lambda self, *a, **k: _init(self, *a, **{{**k, 'device': 'gpu'}})

# 3. 设置参数并运行
sys.argv = [
    './backend/main.py',
    '-i', '{input_path}/input-3m.mp4',
    '-o', '{output_path}/output-3m.mp4',
    '--inpaint-mode', 'sttn-det',
    '--subtitle-area-coords', '37','153','1516','1885',
    '--subtitle-area-coords', '892','1080','0','1919',
]

runpy.run_path('./backend/main.py', run_name='__main__')
"""

with open('tmp_patch_run.py', 'w') as f:
    f.write(patch_code)

!{exe_path}/.venv/bin/python tmp_patch_run.py

if os.path.exists('tmp_patch_run.py'):
    os.remove('tmp_patch_run.py')

/kaggle/temp/video-subtitle-remover
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.

📢 Tips: QFluentWidgets Pro is now released. Click https://qfluentwidgets.com/pages/pro to learn more about it.

ONNX provider: TensorrtExecutionProvider not supported, skipped.
Detected ONNX provider: CUDAExecutionProvider
Subtitle Area: [[37, 153, 1516, 1885], [892, 1080, 0, 1919]]
Processing block: All
Subtitle Removing:   0%|                            | 0/5529 [00:00<?, ?frame/s]Subtitle removal model: STTN Detection (GPU)
Subtitle detection model: Precise (CUDAExecutionProvider)
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
Subtitle Finding:   0%|                             | 0/5529 [00:00<?, ?frame/s][Processing] Detecting subtitles...
The Paddle

In [ ]:
# @title 直接运行
# %cd {pro_path}
# !/content/.venv/bin/python ./backend/main.py -i {input_path}/input-3s.mp4 -o {output_path}/output-3s.mp4 --inpaint-mode sttn-det